In [1]:
import torch
import sys
sys.path.append("../")
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import plot_utils
from plot_utils import plot_comparison, plot_stats
from IPython.display import display, Markdown
import pandas as pd
import tempfile
from cumulant_analyzer import calculate_cumulants, to_numpy, shuffle_tokens

from transformers.utils.logging import disable_progress_bar
disable_progress_bar()

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

%matplotlib inline

/home/karthikv/.conda/envs/llmenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Disabled automatic differentiation


In [2]:
model_name = "gpt2-large"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")  
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=False)   

## Utils

In [3]:
def calculate_cumulants_single_layer(logits, beta):
    """
    Calculate cumulants for a single layer at a given inverse temperature.
    
    Args:
        logits: Tensor of shape (seq_len, vocab_size) - raw logits
        beta: Inverse temperature (1/T)
    
    Returns:
        Dictionary with cumulants and related statistics
    """
    with torch.no_grad():
        # Apply temperature scaling
        scaled_logits = logits * beta
        probs = torch.softmax(scaled_logits, dim=-1)
        
        # Calculate combined probabilities (average over sequence)
        probs_com = probs.mean(dim=0)  # Shape: (vocab_size,)
        logits_com = torch.log(probs_com + 1e-12)
        logits_com = logits_com - (probs_com * logits_com).sum()
        
        # Calculate delta (difference from combined logits)
        delta = logits_com.unsqueeze(0) - scaled_logits  # Shape: (seq_len, vocab_size)
        delta_centered = delta - (probs * delta).sum(dim=-1, keepdim=True)
        
        # Calculate raw moments up to 8th order
        moments = {}
        for k in range(1, 9):
            moments[k] = (probs * delta_centered.pow(k)).sum(dim=-1)  # Shape: (seq_len,)
        
        # Cumulant formulas
        cumulant_formulas = {
            2: lambda m: m[2],
            3: lambda m: m[3],
            4: lambda m: m[4] - 3 * m[2] ** 2,
            5: lambda m: m[5] - 10 * m[2] * m[3],
            6: lambda m: m[6] - 15 * m[2] * m[4] - 10 * m[3] ** 2 + 30 * m[2] ** 3,
            7: lambda m: m[7] - 21 * m[2] * m[5] - 35 * m[3] * m[4] + 210 * m[2] ** 2 * m[3],
            8: lambda m: (
                m[8] - 28 * m[2] * m[6] - 56 * m[3] * m[5] + 420 * m[2] ** 2 * m[4]
                - 35 * m[4] ** 2 + 560 * m[2] * m[3] ** 2 - 630 * m[2] ** 4
            ),
        }
        
        # Calculate cumulants (shape: (7, seq_len) for cumulants 2-8)
        cumulants = torch.stack([
            cumulant_formulas[k](moments) for k in range(2, 9)
        ])
        
        # Calculate normalized cumulants
        normalization_factors = torch.tensor(
            [2, 6, 24, 120, 720, 5040, 40320], 
            device=cumulants.device
        ).unsqueeze(1)  # Shape: (7, 1)
        
        normalized_cumulants = cumulants / normalization_factors
        
        # Calculate entropy
        entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=-1)  # Shape: (seq_len,)
        
        # Calculate KL divergence
        kld = (probs * (torch.log(probs + 1e-12) - torch.log(probs_com + 1e-12))).sum(dim=-1)
        
        return {
            'beta': beta,
            'temperature': 1.0 / beta,
            'cumulants': cumulants.mean(dim=-1).cpu().numpy(),  # Average over sequence
            'normalized_cumulants': normalized_cumulants.mean(dim=-1).cpu().numpy(),
            'entropy': entropy.mean().cpu().numpy(),
            'kld': kld.mean().cpu().numpy(),
            'entropy_com': -(probs_com * torch.log(probs_com + 1e-12)).sum().cpu().numpy()
        }

def analyze_temperature_sweep_precomputed(logits, shuffled_logits, conditions, beta_min=0.1, beta_max=5.0, n_steps=50, verbose = False):
    """
    Analyze cumulants across a range of inverse temperatures for pre-computed structured and shuffled logits.
    
    Args:
        logits: Tensor of shape (seq_len, vocab_size) - structured logits
        shuffled_logits: Tensor of shape (seq_len, vocab_size) - shuffled logits
        beta_min: Minimum inverse temperature
        beta_max: Maximum inverse temperature  
        n_steps: Number of temperature steps
    
    Returns:
        DataFrame with results for each temperature and condition
    """
    # Create range of inverse temperatures
    betas = np.linspace(beta_min, beta_max, n_steps)
    
    results = []
    
    if verbose: print(f"Analyzing {n_steps} temperature points from β={beta_min} to β={beta_max}")
    
    # Analyze structured logits
    if verbose: print("Processing structured logits...")
    for i, beta in enumerate(betas):
        if (i + 1) % 10 == 0:
            if verbose: print(f"  Structured: {i+1}/{n_steps} (β={beta:.2f}, T={1/beta:.2f})")
            
        result = calculate_cumulants_single_layer(logits, beta)
        result['condition'] = conditions[0]
        results.append(result)
    
    # Analyze shuffled logits
    if verbose: print("Processing shuffled logits...")
    for i, beta in enumerate(betas):
        if (i + 1) % 10 == 0:
            if verbose: print(f"  Shuffled: {i+1}/{n_steps} (β={beta:.2f}, T={1/beta:.2f})")
            
        result = calculate_cumulants_single_layer(shuffled_logits, beta)
        result['condition'] = conditions[1]
        results.append(result)
    
    # Convert to DataFrame
    df_results = pd.DataFrame(results)
    
    # Expand cumulants and normalized_cumulants into separate columns
    cumulant_cols = [f'kappa_{i+2}' for i in range(7)]  # κ₂ through κ₈
    norm_cumulant_cols = [f'norm_kappa_{i+2}' for i in range(7)]
    
    # Add individual cumulant columns
    for i, col in enumerate(cumulant_cols):
        df_results[col] = df_results['cumulants'].apply(lambda x: x[i])
    
    for i, col in enumerate(norm_cumulant_cols):
        df_results[col] = df_results['normalized_cumulants'].apply(lambda x: x[i])
    
    # Drop the array columns
    df_results = df_results.drop(['cumulants', 'normalized_cumulants'], axis=1)
    
    return df_results


def plot_temperature_analysis(df_results, save_path=None):
    """
    Plot cumulants vs temperature for both structured and shuffled conditions.
    
    Args:
        df_results: DataFrame from analyze_temperature_sweep
        save_path: Optional path to save the plot
    """
    # Check if we have both conditions
    conditions = df_results['condition'].unique()
    has_both = len(conditions) == 2
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))  # Increased height slightly for legend space
    axes = axes.flatten()
    
    # Colors for different conditions
    colors = {conditions[0]: 'blue', conditions[1]: 'red'}
    line_styles = {conditions[0]: '-', conditions[1]: '-'}
    
    # Plot normalized cumulants κ₂ through κ₈
    cumulant_names = [f'norm_kappa_{i+2}' for i in range(7)]
    cumulant_labels = [f'$κ_{i+2}$' for i in range(7)]
    
    for i, (col, label) in enumerate(zip(cumulant_names, cumulant_labels)):
        ax = axes[i+1]
        
        for condition in conditions:
            condition_data = df_results[df_results['condition'] == condition]
            ax.plot(condition_data['temperature'], condition_data[col], 
                   color=colors[condition], linestyle=line_styles[condition], 
                   linewidth=2, label=condition)
        
        ax.set_xlabel('Temperature (T = 1/β)')
        ax.set_ylabel(f'Normalized {label}')
        ax.set_title(f'{label}')
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
    
    # Plot entropy in the first subplot
    ax = axes[0]
    for condition in conditions:
        condition_data = df_results[df_results['condition'] == condition]
        ax.plot(condition_data['temperature'], condition_data['entropy'], 
               color=colors[condition], linestyle=line_styles[condition], 
               linewidth=2, label=f'{condition} S')
        ax.plot(condition_data['temperature'], condition_data['entropy_com'], 
               color=colors[condition], linestyle=':', 
               linewidth=2, label=f'{condition} S($\mu$)')
    
    ax.set_xlabel('Temperature (T = 1/β)')
    ax.set_ylabel('Entropy')
    ax.set_title('Entropy vs Temperature')
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    
    plt.tight_layout()
    
    # Add legends at the bottom of the figure
    if has_both:
        # Get handles and labels from the entropy plot (most comprehensive)
        handles, labels = axes[0].get_legend_handles_labels()
        
        # Create a single legend at the bottom center
        fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.02), 
                  ncol=len(labels), frameon=True, fancybox=True, shadow=True)
        
        # Adjust layout to make room for the legend
        plt.subplots_adjust(bottom=0.1)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

## Analysis

### Shuffled vs Structured

In [ ]:
ds = load_dataset("NeelNanda/pile-10k")['train']
filtered_indices = np.load('filtered_indices.npy')
max_length = 256
device = torch.device('cuda');
test_indx = 686;


test_sequence = ds['text'][filtered_indices[test_indx]]

input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens = False, \
                 return_tensors = "pt", max_length = max_length, truncation = True).to(device)
outputs = model(input_ids, output_hidden_states=True)
logits = outputs.logits.squeeze()
input_ids = shuffle_tokens(input_ids)
outputs = model(input_ids, output_hidden_states=True)
shuffled_logits = outputs.logits.squeeze()
df_results = analyze_temperature_sweep_precomputed(logits, shuffled_logits,conditions = ["Structured", "Shuffled"],
                                                  beta_min=0.1, beta_max=10.0, n_steps=50)

# Plot all cumulants comparison
plot_temperature_analysis(df_results)

### Pile vs Math

In [ ]:
def choose_random_index_from_topic(input_topic):
    filered_topics = [item['pile_set_name'] for i, item in enumerate(ds['meta']) if i in filtered_indices]
    indices = [i for i, topic in enumerate(filered_topics) if topic == input_topic]
    # print(indices)
    return np.random.choice(indices)

In [ ]:
topic1 = "DM Mathematics"
index1 = choose_random_index_from_topic(topic1)
test_sequence = ds['text'][filtered_indices[index1]]

input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens = False, \
                 return_tensors = "pt", max_length = max_length, truncation = True).to(device)
outputs = model(input_ids, output_hidden_states=True)
logits1 = outputs.logits.squeeze()

topic2 = "Pile-CC"
index2 = choose_random_index_from_topic(topic2)
test_sequence = ds['text'][filtered_indices[index2]]
input_ids = tokenizer.encode(test_sequence.strip(), add_special_tokens = False, \
                 return_tensors = "pt", max_length = max_length, truncation = True).to(device)
outputs = model(input_ids, output_hidden_states=True)
logits2 = outputs.logits.squeeze()


df_results = analyze_temperature_sweep_precomputed(logits1, logits2, conditions = [topic1, topic2], 
                                                  beta_min=0.1, beta_max=10.0, n_steps=100)

# Plot all cumulants comparison
plot_temperature_analysis(df_results)